In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_project.gold;

In [0]:
from pyspark.sql.functions import col, sequence, explode, to_date, year, month, quarter, when

# Get min and max dates from encounters
date_range = spark.sql("""
    SELECT 
        MIN(start) AS min_date,
        MAX(stop) AS max_date
    FROM medical_project.silver.encounters
""")

dates = date_range.select(
    explode(
        sequence(
            to_date(col("min_date")),
            to_date(col("max_date"))
        )
    ).alias("date")
)

# Create date dimension
dim_date = dates.withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn(
        "half_year",
        when(col("month") <= 6, 1).otherwise(2)
    )

In [0]:
display(dim_date)

In [0]:
# Save table
dim_date.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.dim_date")